In [1]:
import pandas as pd

df = pd.read_csv("../data/raw/compas-scores-two-years.csv")
print(df.shape)
df.head()

(7214, 53)


,id,name,first,last,compas_screening_date,sex,dob,age,age_cat,race,...,v_decile_score,v_score_text,v_screening_date,in_custody,out_custody,priors_count.1,start,end,event,two_year_recid
0,1,miguel hernandez,miguel,hernandez,2013-08-14,Male,1947-04-18,69,Greater than 45,Other,...,1,Low,2013-08-14,2014-07-07,2014-07-14,0,0,327,0,0
1,3,kevon dixon,kevon,dixon,2013-01-27,Male,1982-01-22,34,25 - 45,African-American,...,1,Low,2013-01-27,2013-01-26,2013-02-05,0,9,159,1,1
2,4,ed philo,ed,philo,2013-04-14,Male,1991-05-14,24,Less than 25,African-American,...,3,Low,2013-04-14,2013-06-16,2013-06-16,4,0,63,0,1
3,5,marcu brown,marcu,brown,2013-01-13,Male,1993-01-21,23,Less than 25,African-American,...,6,Medium,2013-01-13,NaN,NaN,1,0,1174,0,0
4,6,bouthy pierrelouis,bouthy,pierrelouis,2013-03-26,Male,1973-01-22,43,25 - 45,Other,...,1,Low,2013-03-26,NaN,NaN,2,0,1102,0,0


In [2]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7214 entries, 0 to 7213
Data columns (total 53 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   id                       7214 non-null   int64  
 1   name                     7214 non-null   str    
 2   first                    7214 non-null   str    
 3   last                     7214 non-null   str    
 4   compas_screening_date    7214 non-null   str    
 5   sex                      7214 non-null   str    
 6   dob                      7214 non-null   str    
 7   age                      7214 non-null   int64  
 8   age_cat                  7214 non-null   str    
 9   race                     7214 non-null   str    
 10  juv_fel_count            7214 non-null   int64  
 11  decile_score             7214 non-null   int64  
 12  juv_misd_count           7214 non-null   int64  
 13  juv_other_count          7214 non-null   int64  
 14  priors_count             7214 non-n

In [3]:
df.isna().sum()

id                            0
name                          0
first                         0
last                          0
compas_screening_date         0
sex                           0
dob                           0
age                           0
age_cat                       0
race                          0
juv_fel_count                 0
decile_score                  0
juv_misd_count                0
juv_other_count               0
priors_count                  0
days_b_screening_arrest     307
c_jail_in                   307
c_jail_out                  307
c_case_number                22
c_offense_date             1159
c_arrest_date              6077
c_days_from_compas           22
c_charge_degree               0
c_charge_desc                29
is_recid                      0
r_case_number              3743
r_charge_degree            3743
r_days_from_arrest         4898
r_offense_date             3743
r_charge_desc              3801
r_jail_in                  4898
r_jail_o

In [4]:
df[["age", "sex", "race", "priors_count", "c_charge_degree", "decile_score", "two_year_recid"]].describe(include="all")

,age,sex,race,priors_count,c_charge_degree,decile_score,two_year_recid
count,7214.000000,7214,7214,7214.000000,7214,7214.000000,7214.000000
unique,NaN,2,6,NaN,2,NaN,NaN
top,NaN,Male,African-American,NaN,F,NaN,NaN
freq,NaN,5819,3696,NaN,4666,NaN,NaN
mean,34.817993,NaN,NaN,3.472415,NaN,4.509565,0.450652
std,11.888922,NaN,NaN,4.882538,NaN,2.856396,0.497593
min,18.000000,NaN,NaN,0.000000,NaN,1.000000,0.000000
25%,25.000000,NaN,NaN,0.000000,NaN,2.000000,0.000000
50%,31.000000,NaN,NaN,2.000000,NaN,4.000000,0.000000
75%,42.000000,NaN,NaN,5.000000,NaN,7.000000,1.000000


# 01. Data Exploration

### ProPublica COMPAS Dataset Summary

* **Source:** ProPublica's COMPAS dataset (Broward County, Florida), available on [GitHub](https://github.com/propublica/compas-analysis).
* **Shape:** The dataset contains **7,214 rows** and **53 columns**.

---

### Key Dataset Columns Overview

* **`age`**: The age of the individual in years at the time of screening, ranging from **18 to 96 years old** (mean: 34.8 years).
* **`sex`**: The sex of the individual as recorded in the dataset (**5,819 Male**, **1,395 Female**).
* **`race`**: The self-reported racial background of the individual across **6 categories** (**3,696 African-American**, **2,454 Caucasian**, **509 Hispanic**, **377 Other**, **145 Asian**, **33 Native American**).
* **`priors_count`**: The number of prior offenses (charges) recorded for the individual before this screening (ranging from 0 to 38).
* **`c_charge_degree`**: The severity of the current charge (**4,666 Felony (`F`)**, **2,548 Misdemeanor (`M`)**).
* **`decile_score`**: An algorithmic risk score assigned by the COMPAS system on a scale from 1 (lowest risk) to 10 (highest risk).
* **`two_year_recid`**: A binary indicator (`0` = No, `1` = Yes) showing whether the individual was rearrested within two years of screening.

---

### Prediction (`decile_score`) vs. Reality (`two_year_recid`)

* **`decile_score` (The Prediction):** This is a **risk assessment score** generated by COMPAS. It represents an algorithmic estimate of how likely an individual is to be rearrested in the future, ranked relative to other defendants on a 1-10 scale.
* **`two_year_recid` (The Ground Truth):** This is the **actual outcome variable** observed over a two-year follow-up period. A value of `1` indicates that the person was rearrested within two years, while `0` means they were not.

Comparing these two columns allows us to evaluate the accuracy, fairness, and error rates (such as false positives and false negatives) of the COMPAS prediction model. Note that because `two_year_recid` tracks rearrests rather than proven crimes, biases in policing may influence this ground truth label.

---

### Missing Values Analysis

Several columns contain missing values:

* **Current charge/arrest details:** `c_arrest_date` has 6,077 missing values, `c_offense_date` has 1,159 missing, and `days_b_screening_arrest`, `c_jail_in`, and `c_jail_out` each have 307 missing values.
* **Rearrest details (`r_*` columns):** `r_case_number`, `r_charge_degree`, and `r_offense_date` have 3,743 missing values, while `r_days_from_arrest`, `r_jail_in`, and `r_jail_out` have 4,898 missing. These columns describe a rearrest, so they are only filled in for people with a recorded rearrest.
* **Violent rearrest details:** `violent_recid` is completely empty (7,214 missing) and can be dropped, whereas `vr_case_number` and related columns are missing for 6,395 rows where no violent rearrest was recorded.

The columns needed for the core fairness audit (`race`, `sex`, `age`, `decile_score`, `two_year_recid`) have zero missing values. The 307 rows with a missing `days_b_screening_arrest` will be dropped during cleaning, because the screening-date filter cannot be satisfied when the value is missing.

---

### Columns Not to Use as Model Inputs

* Columns starting with `r_`, `vr_`, and `is_recid` describe what happened **after** the screening. Using them to predict the outcome would be **data leakage**: the model would look excellent but would be using information that would not exist at prediction time.
* `decile_score` is COMPAS's own prediction, so it is treated as a comparison baseline, not as an input feature.
* `race` is excluded from model features but kept aside so fairness can be measured across groups.

---

### Key Research Questions

1. Does the COMPAS risk score (`decile_score`) exhibit disparate impact or differential predictive accuracy across different racial groups (e.g., African-American vs. Caucasian defendants)?
2. Are false positive rates (predicting high risk for someone who is not rearrested) or false negative rates significantly higher for specific demographic groups?
3. Does a specific `decile_score` value correspond to the same actual rearrest probability (`two_year_recid`) across different races and sexes?
4. Can models trained without race as a feature still show a gap between groups?